# Deep Learning / NLP Text Classifier - Sanika Hajare
- Task: Sentiment Analysis (IMDB style synthetic + TF-IDF + PyTorch NN)
- Dropout + BatchNorm + EarlyStopping + Training Curves + Inference

In [ ]:
import pandas as pd, numpy as np, random, re
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt
np.random.seed(42); random.seed(42); torch.manual_seed(42)

# Synthetic sentiment dataset - Rabtech acceptable
pos = ['I love this movie it is fantastic','Great product amazing experience','Excellent service loved it','Superb quality highly recommend','Wonderful and delightful']*200
neg = ['I hate this terrible product','Worst experience awful service','Bad quality disappointing','Horrible movie waste of time','Poor and useless item']*200
texts = pos+neg
labels = [1]*len(pos) + [0]*len(neg)
df = pd.DataFrame({'text':texts, 'label':labels}).sample(frac=1, random_state=42).reset_index(drop=True)
print(df['label'].value_counts())
df.head()


In [ ]:
# Tokenize & Vectorize - TF-IDF
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1,2))
X = vectorizer.fit_transform(df['text']).toarray()
y = df['label'].values
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# Dataset
class TextDataset(Dataset):
 def __init__(self,x,y): self.x=torch.FloatTensor(x); self.y=torch.FloatTensor(y)
 def __len__(self): return len(self.x)
 def __getitem__(self,i): return self.x[i], self.y[i]
train_ds = TextDataset(X_train, y_train)
test_ds = TextDataset(X_test, y_test)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64)


In [ ]:
# Multi-layer NN with Dropout + BatchNorm
class SentimentNN(nn.Module):
 def __init__(self, input_dim):
 super().__init__()
 self.net = nn.Sequential(
 nn.Linear(input_dim, 256),
 nn.BatchNorm1d(256),
 nn.ReLU(),
 nn.Dropout(0.5),
 nn.Linear(256, 128),
 nn.BatchNorm1d(128),
 nn.ReLU(),
 nn.Dropout(0.3),
 nn.Linear(128, 64),
 nn.BatchNorm1d(64),
 nn.ReLU(),
 nn.Dropout(0.3),
 nn.Linear(64, 1)
 )
 def forward(self,x): return self.net(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SentimentNN(X_train.shape[1]).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2)
print(model)


In [ ]:
# Train with Early Stopping
train_losses, val_losses, train_accs, val_accs = [], [], [], []
best_val_loss = float('inf'); patience=4; patience_counter=0; best_state=None
EPOCHS=20

for epoch in range(EPOCHS):
 model.train(); total_loss=0; correct=0; total=0
 for xb,yb in train_loader:
 xb,yb = xb.to(device), yb.to(device)
 optimizer.zero_grad()
 out = model(xb).squeeze()
 loss = criterion(out, yb)
 loss.backward(); optimizer.step()
 total_loss+=loss.item()
 preds = (torch.sigmoid(out)>0.5).float()
 correct+=(preds==yb).sum().item(); total+=yb.size(0)
 train_loss=total_loss/len(train_loader); train_acc=correct/total
 
 model.eval(); v_loss=0; v_correct=0; v_total=0
 with torch.no_grad():
 for xb,yb in test_loader:
 xb,yb = xb.to(device), yb.to(device)
 out = model(xb).squeeze()
 loss = criterion(out, yb); v_loss+=loss.item()
 preds = (torch.sigmoid(out)>0.5).float()
 v_correct+=(preds==yb).sum().item(); v_total+=yb.size(0)
 val_loss=v_loss/len(test_loader); val_acc=v_correct/v_total
 scheduler.step(val_loss)
 train_losses.append(train_loss); val_losses.append(val_loss); train_accs.append(train_acc); val_accs.append(val_acc)
 print(f"Epoch {epoch+1}/{EPOCHS} Train Loss:{train_loss:.4f} Acc:{train_acc:.4f} | Val Loss:{val_loss:.4f} Acc:{val_acc:.4f}")
 if val_loss < best_val_loss:
 best_val_loss=val_loss; best_state=model.state_dict().copy(); patience_counter=0
 else:
 patience_counter+=1
 if patience_counter>=patience:
 print(f"Early stopping at epoch {epoch+1}"); break
model.load_state_dict(best_state)
print(f"Best Val Loss: {best_val_loss:.4f}")


In [ ]:
# Plot Loss & Accuracy curves
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(train_losses,label='Train Loss'); plt.plot(val_losses,label='Val Loss'); plt.legend(); plt.title('Loss Convergence'); plt.xlabel('Epoch')
plt.subplot(1,2,2)
plt.plot(train_accs,label='Train Acc'); plt.plot(val_accs,label='Val Acc'); plt.legend(); plt.title('Accuracy Convergence'); plt.xlabel('Epoch')
plt.savefig('training_curves.png'); plt.show()
print('Saved: training_curves.png')


In [ ]:
# Evaluation + Inference on unseen samples
model.eval()
with torch.no_grad():
 X_test_t=torch.FloatTensor(X_test).to(device)
 probs=torch.sigmoid(model(X_test_t).squeeze()).cpu().numpy()
 preds=(probs>0.5).astype(int)
print(classification_report(y_test,preds))
print(f"Accuracy: {accuracy_score(y_test,preds):.4f}")

# Unseen samples inference with confidences
samples = ["I absolutely love this amazing product","This is the worst terrible experience","Great fantastic wonderful service","Horrible disappointing bad quality","I love it but also hate some parts"]
sample_vec = vectorizer.transform(samples).toarray()
sample_t = torch.FloatTensor(sample_vec).to(device)
with torch.no_grad():
 conf = torch.sigmoid(model(sample_t).squeeze()).cpu().numpy()
for txt, p in zip(samples, conf):
 label = 'POSITIVE' if p>0.5 else 'NEGATIVE'
 print(f"{label} ({p:.3f}) -> {txt}")


In [ ]:
# Save model
import joblib
torch.save(model.state_dict(), 'nlp_sentiment_pytorch.pt')
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')
print('Saved: nlp_sentiment_pytorch.pt + tfidf_vectorizer.pkl - Ready for Rabtech proof')
